In [ ]:

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. LOAD DATASET

df = pd.read_csv('imdb_data_cleaned.csv')
data = pd.read_csv('imdb_data_cleaned.csv')

# Fill missing values
data["cast"] = data["cast"].fillna("").str.lower()
data["genres"] = data["genres"].fillna("").str.lower()

# Convert year and rating into numeric values
data["releaseyear"] = pd.to_numeric(data["releaseyear"], errors="coerce")

data["imdbrating"] = pd.to_numeric(data["imdbrating"], errors="coerce")

# 2. FILTER MOVIES FROM 2000 TO 2024

movies = data[
    (data["type"].str.lower() == "movie") &
    (data["releaseyear"] >= 2000) &
    (data["releaseyear"] <= 2024) &
    (data["imdbrating"].notna())
].copy()

# 3. TRAIN TF-IDF MODEL

vectorizer = TfidfVectorizer()

genre_matrix = vectorizer.fit_transform(movies["genres"])

print("TF-IDF model trained successfully!")


# Common function to sort by rating and year

def sort_movies(result):
    return result.sort_values(
        by=["imdbrating", "releaseyear"],
        ascending=[False, False]
    ).head(10)


# 4. TOP 10 MOVIES BY RATING

print("\n========== TOP 10 MOVIES ==========")

top_movies = sort_movies(movies)

print(
    top_movies[
        ["title", "imdbrating", "releaseyear", "genres"]
    ].to_string(index=False)
)


# 5. SEARCH BY GENRE

def search_genre(genre):

    genre = genre.lower().strip()

    # Convert user input into TF-IDF
    user_vector = vectorizer.transform([genre])

    # Calculate cosine similarity
    similarity = cosine_similarity(
        user_vector, genre_matrix
    ).flatten()

    result = movies[similarity > 0.2].copy()

    result["similarity"] = similarity[similarity > 0.2]

    # Sort by rating first, then newest year
    result = result.sort_values(
        by=["imdbrating", "releaseyear"],
        ascending=[False, False]
    )

    return result.head(10)


# 6. SEARCH BY ACTOR

def search_actor(actor):

    actor = actor.lower().strip()

    result = movies[
        movies["cast"].str.contains(actor, regex=False, na=False)
    ]

    # Sort by rating first, then newest year
    result = result.sort_values(
        by=["imdbrating", "releaseyear"],
        ascending=[False, False]
    )

    return result.head(10)


# 7. MENU

while True:

    print("\n================================")
    print("       NETFLIX SEARCH SYSTEM")
    print("================================")

    print("1. Top 10 Movies")
    print("2. Search by Genre")
    print("3. Search by Actor")
    print("q. Quit")

    choice = input("\nEnter your choice: ").strip()

    # TOP 10 MOVIES

    if choice == "1":

        result = sort_movies(movies)

        print("\nTop 10 Movies by IMDb Rating:")

        print(
            result[
                ["title", "imdbrating", "releaseyear", "genres"]
            ].to_string(index=False)
        )

    # SEARCH BY GENRE

    if choice == "2":

        genre = input("Enter genre: ")

        result = search_genre(genre)

        if result.empty:
            print("\nNo movies found for this genre.")

        else:
            print("\nTop movies for genre:", genre)

            print(
                result[
                    ["title", "imdbrating", "releaseyear", "genres"]
                ].to_string(index=False)
            )

    # SEARCH BY ACTOR

    elif choice == "3":

        actor = input("Enter actor name: ")

        result = search_actor(actor)

        if result.empty:
            print("\nNo movies found for this actor.")

        else:
            print("\nTop movies featuring:", actor)

            print(
                result[
                    ["title", "imdbrating", "releaseyear", "genres"]
                ].to_string(index=False)
            )

    # QUIT

    elif choice.lower() == "q":

        print("\nProgram ended.")
        break

    else:
        print("\nInvalid choice! Please try again.")
        input("Press Enter to exit...")
        import traceback

try:
    # -----------------------------------------
    # PASTE ALL YOUR ORIGINAL CODE HERE
    # (Imports, dataset loading, ML models, etc.)
    # -----------------------------------------
    print("Script finished successfully!")

except Exception as e:
    print("\n--- AN ERROR OCCURRED ---")
    traceback.print_exc()

finally:
    # This ensures the window stays open whether it succeeds or fails
    input("\nPress Enter to close this window...")

FileNotFoundError: [Errno 2] No such file or directory: 'imdb_data_cleaned.csv'